# Linear Algebra in Transformers

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 01.01–01.09 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/10_linear_algebra_in_transformers.ipynb)

---

## 🎯 Learning Objective

Walk through the transformer architecture end-to-end through a purely linear-algebra lens: Q/K/V projections, scaled dot-product attention, multi-head concatenation, and residual connections.

---

## 📐 Theory

This notebook is the synthesis of Module 01: we walk through one transformer block end-to-end,
naming which linear-algebra concept powers each step.

### Step 1 — Embeddings ([01.01](01_vectors_and_spaces.ipynb))

Each token is looked up as a row of an embedding matrix, becoming a vector in
$\mathbb{R}^{d_{\text{model}}}$.

### Step 2 — Q/K/V projections ([01.02](02_matrix_operations.ipynb), [01.03](03_linear_transformations.ipynb))

Three learned weight matrices linearly transform every token embedding $\mathbf{x}$ into a
query, key, and value:

$$\mathbf{q} = W_Q\mathbf{x} \qquad \mathbf{k} = W_K\mathbf{x} \qquad \mathbf{v} = W_V\mathbf{x}$$

### Step 3 — Scaled dot-product attention ([01.01](01_vectors_and_spaces.ipynb), [01.09](09_tensor_operations.ipynb))

Stacking every token's query/key/value as rows of matrices $Q,K,V$ (shape
`(seq_len, d_k)`), attention is:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

$QK^\top$ computes *every pairwise dot product* between queries and keys at once — a grid of
cosine-similarity-like scores (Notebook 01.01), scaled by matrix multiplication
(Notebook 01.02). Dividing by $\sqrt{d_k}$ keeps the dot products from growing too large as
dimensionality increases (we derive exactly why in
[Notebook 02.03](../02_Calculus/03_partial_derivatives.ipynb)'s treatment of variance scaling).
The softmax turns scores into a weighted average recipe, and multiplying by $V$ mixes value
vectors according to those weights.

### Step 4 — Multi-head attention ([01.09](09_tensor_operations.ipynb))

Rather than one attention computation in the full $d_{\text{model}}$ dimensions, the model
splits into $h$ heads of dimension $d_k = d_{\text{model}}/h$, runs attention independently per
head (a batched tensor contraction), then concatenates and linearly re-projects the results.

### Step 5 — Residual connections ([01.03](03_linear_transformations.ipynb))

Each sub-layer's output is *added back* to its input: $\mathbf{x}_{\text{out}} = \mathbf{x} +
\text{Sublayer}(\mathbf{x})$. In transformation terms, this is "identity transformation plus a
learned correction" — it guarantees a direct gradient path all the way back to the input, no
matter how deep the network, because the identity part of $\mathbf{x}_{\text{out}}$ always
carries gradient straight through.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

An attention weight matrix is literally a heatmap of "how much does query position $i$ attend
to key position $j$" — visualizing it makes the abstract $QK^\top$ formula into a picture of
which tokens are talking to which.

In [ ]:
# Visualizing an attention weight matrix as a token-to-token heatmap
import numpy as np
import matplotlib.pyplot as plt

def softmax_rows(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len, d_k = len(tokens), 8
rng = np.random.default_rng(3)

Q = rng.standard_normal((seq_len, d_k))
K = rng.standard_normal((seq_len, d_k))
scores = (Q @ K.T) / np.sqrt(d_k)
weights = softmax_rows(scores)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap="viridis")
ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
ax.set_xlabel("Key position (attended TO)"); ax.set_ylabel("Query position (attending FROM)")
ax.set_title(r"softmax($QK^T / \sqrt{d_k}$): each row sums to 1")
plt.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()
print("Each row sums to 1 (it's a probability distribution over 'where to look'):", np.allclose(weights.sum(axis=1), 1.0))

## 🐍 Implementation from Scratch

A complete single-head scaled dot-product attention function, then multi-head attention built
by batching that single-head computation across a head axis using the einsum patterns from
[Notebook 01.09](09_tensor_operations.ipynb).

In [ ]:
# Full scaled dot-product attention + multi-head attention from scratch
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    # Q, K, V: shape (..., seq_len, d_k). Returns (..., seq_len, d_v) and the attention weights.
    d_k = Q.shape[-1]
    scores = np.einsum('...id,...jd->...ij', Q, K) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = np.einsum('...ij,...jd->...id', weights, V)
    return output, weights

def multi_head_attention(x, Wq, Wk, Wv, Wo, n_heads):
    # x: (seq_len, d_model). Wq/Wk/Wv/Wo: (d_model, d_model). Splits into n_heads internally.
    seq_len, d_model = x.shape
    d_k = d_model // n_heads

    Q, K, V = x @ Wq, x @ Wk, x @ Wv                     # (seq_len, d_model) each

    def split_heads(M):
        return M.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)   # -> (n_heads, seq_len, d_k)

    Qh, Kh, Vh = split_heads(Q), split_heads(K), split_heads(V)
    out_heads, _ = scaled_dot_product_attention(Qh, Kh, Vh)           # (n_heads, seq_len, d_k)

    concatenated = out_heads.transpose(1, 0, 2).reshape(seq_len, d_model)  # merge heads back
    return concatenated @ Wo                                               # final linear re-projection

rng = np.random.default_rng(0)
seq_len, d_model, n_heads = 6, 16, 4
x = rng.standard_normal((seq_len, d_model)) * 0.1
Wq, Wk, Wv, Wo = (rng.standard_normal((d_model, d_model)) * 0.1 for _ in range(4))

output = multi_head_attention(x, Wq, Wk, Wv, Wo, n_heads)
print("Input shape :", x.shape)
print("Output shape:", output.shape, " (matches input -- ready for the residual connection)")

# Residual connection: identity + learned correction
x_out = x + output
print("\nResidual output x + Sublayer(x), shape:", x_out.shape)
print("If the sublayer output were all zeros, x_out would exactly equal x (identity preserved).")

## 🤖 Why This Matters for AI

Everything above **is** the AI connection — this notebook builds a real (if small) transformer
block. To close the loop, we confirm our from-scratch multi-head attention exactly matches
PyTorch's reference implementation, `torch.nn.functional.scaled_dot_product_attention`, given
identical inputs and weights.

In [ ]:
# Cross-checking our from-scratch attention against PyTorch's reference implementation
import numpy as np
import torch
import torch.nn.functional as F

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = np.einsum('...id,...jd->...ij', Q, K) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return np.einsum('...ij,...jd->...id', weights, V), weights

rng = np.random.default_rng(0)
n_heads, seq_len, d_k = 4, 6, 8
Q = rng.standard_normal((n_heads, seq_len, d_k)).astype(np.float32)
K = rng.standard_normal((n_heads, seq_len, d_k)).astype(np.float32)
V = rng.standard_normal((n_heads, seq_len, d_k)).astype(np.float32)

our_output, _ = scaled_dot_product_attention(Q, K, V)

torch_output = F.scaled_dot_product_attention(
    torch.from_numpy(Q), torch.from_numpy(K), torch.from_numpy(V)
).numpy()

print("Our from-scratch output shape :", our_output.shape)
print("PyTorch reference output shape:", torch_output.shape)
print("Outputs match PyTorch's own implementation:", np.allclose(our_output, torch_output, atol=1e-5))
print("\nEvery concept from this module -- vectors, matrices, transformations, eigenstructure,")
print("SVD, and tensors -- combined into the exact mechanism inside every modern LLM.")

## 🏋️ Exercises

### Warm-up
1. For the 6-token attention weight matrix from the Visual Intuition cell, print out which single key position each query attends to MOST STRONGLY (`np.argmax` per row), and explain in words what that row of the matrix represents.

### Practice
2. Modify `multi_head_attention` to also return the per-head attention weights, then visualize all 4 heads' weight matrices side by side as heatmaps (as in the Visual Intuition cell) for the same input. Do different heads attend differently?

### Challenge
3. Implement a CAUSAL version of `scaled_dot_product_attention` (mask out attention to future positions, as previewed in Notebook 00.02's causal-mask example) by adding $-\infty$ to disallowed scores before the softmax. Verify row sums still equal 1, verify position $i$ never attends to position $j>i$, and compare your masked output against `torch.nn.functional.scaled_dot_product_attention(..., is_causal=True)`.

---

## 📚 Further Reading
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)
- [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)

---

*Next notebook: [Limits and Continuity](../02_Calculus/01_limits_and_continuity.ipynb)*